# Level 2 on Visual Genome: export -> train AdaMF-MAT -> evaluate\n\nSettings: GPU T4, **Internet On**, add inputs `yenphamnhu/visual-genome` and `dannywu375/visualgenome`. Then **Save Version -> Save & Run All**. A failing stage is logged and later stages are skipped; read `report_level2.txt`.

In [ ]:
# ===== CONFIG (edit only here) =====
N_IMAGES = 8000       # images in the exported VG subset (more = slower epochs)
BUDGET_H = 4.0        # hours of AdaMF-MAT training on VG; epochs are chosen from the measured speed
EPOCH_CAP = 400
LR = 1e-4             # README example uses 1e-4; authors' scripts use 2e-5
SCENE = "/kaggle/input/datasets/yenphamnhu/visual-genome/scene_graphs.json"
IMGS = ["/kaggle/input/datasets/dannywu375/visualgenome/images/VG_100K",
        "/kaggle/input/datasets/dannywu375/visualgenome/images2/VG_100K_2"]
import os, subprocess, time, re, traceback
R, OUT, SRC = "/kaggle/working/AdaMF-MAT", "/kaggle/working", "/kaggle/working/src"
REPORT = f"{OUT}/report_level2.txt"
STATE = {"ok": True, "epochs": 0}
def log(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); open(REPORT, "a").write(s + "\n")
def sh(cmd, cwd=None, timeout=None):
    try:
        p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=timeout)
        return p.returncode, (p.stdout + p.stderr)[-3500:]
    except subprocess.TimeoutExpired as e:
        return -9, "TIMEOUT " + str(e)[-300:]
def stage(name):
    def deco(fn):
        log(f"\n######## {name} ########"); t = time.time()
        try: fn()
        except Exception: STATE["ok"] = False; log("STAGE FAILED:", traceback.format_exc()[-2500:])
        log(f"-- {name} took {time.time()-t:.0f}s"); return fn
    return deco
open(REPORT, "w").write("Level 2 on Visual Genome report\n")
os.makedirs(SRC, exist_ok=True)
print(sh("pip install -q ijson open_clip_torch")[1])

In [ ]:
@stage("1 clone AdaMF-MAT")
def _():
    if not os.path.exists(R):
        log(sh(f"git clone -q https://github.com/zjukg/AdaMF-MAT {R}")[1])
    log(sh(f"ls {R}/benchmarks/MKG-W")[1])

In [ ]:
%%writefile /kaggle/working/src/common.py
"""Shared pieces: data loading, fixed split, query building, metrics.

Everything is seeded and image-level so all methods see the same split (spec section 6).
"""
import random
import re
from collections import defaultdict

import ijson
import numpy as np

# ponytail: hand-picked action list; extend after reading data/queries/predicates.txt
ACTIONS = {"holding", "riding", "playing", "playing with", "eating", "carrying", "flying",
           "pulling", "throwing", "hitting", "catching", "kicking", "watching", "reading",
           "driving", "pushing", "feeding", "petting", "sitting on", "standing on", "walking on"}


def norm(s):
    return re.sub(r"^(a|an|the)\s+|\s+(a|an|the)$", "", s.strip().lower())


def obj_name(o):
    n = o.get("names") or [o.get("name", "")]
    return norm(n[0]) if n and n[0] else ""


def load_images(path, n=None, seed=0, keep_ids=None):
    """Stream scene_graphs.json -> {image_id: {"objs": set(names), "rels": set((s,p,o))}}.

    n: random subset size (seeded, hash-based so no full load needed). keep_ids: restrict to these ids.
    """
    p = None if n is None else n / 108077.0  # ponytail: 108077 = size of full VG file
    out = {}
    with open(path, "rb") as f:
        for img in ijson.items(f, "item"):
            iid = img["image_id"]
            if keep_ids is not None and iid not in keep_ids:
                continue
            if p is not None and random.Random(f"{seed}-{iid}").random() >= p:
                continue
            id2n = {o["object_id"]: obj_name(o) for o in img["objects"]}
            objs = {v for v in id2n.values() if v}
            rels = set()
            for r in img["relationships"]:
                s, o, pr = id2n.get(r["subject_id"]), id2n.get(r["object_id"]), norm(r["predicate"])
                if s and o:
                    rels.add((s, pr, o))
            out[iid] = {"objs": objs, "rels": rels}
    return out


def split_ids(ids, seed=0, frac=(0.8, 0.1, 0.1)):
    """Fixed image-level split. Same ids + seed -> same split. Returns dict name -> list."""
    ids = sorted(ids)
    random.Random(seed).shuffle(ids)
    a, b = int(len(ids) * frac[0]), int(len(ids) * (frac[0] + frac[1]))
    return {"train": ids[:a], "val": ids[a:b], "test": ids[b:]}


def build_queries(images, ids, min_pos=2, max_pos=50, max_q=None, seed=0):
    """Triple queries restricted to `ids` (one split's gallery).

    Query text "s p o"; positives = images in `ids` whose annotation has that action triple.
    """
    idset = set(ids)
    pos = defaultdict(list)
    for i in ids:
        for t in images[i]["rels"]:
            if t[1] in ACTIONS and t[0] != t[2]:
                pos[t].append(i)
    qs = [{"text": f"{s} {p} {o}", "triple": (s, p, o), "pos": sorted(set(v))}
          for (s, p, o), v in pos.items() if min_pos <= len(set(v)) <= max_pos]
    qs.sort(key=lambda q: q["text"])
    random.Random(seed).shuffle(qs)
    return qs[:max_q] if max_q else qs


def metrics(rank_lists, positives, ks=(1, 5, 10)):
    """rank_lists: list of ranked image-id lists; positives: list of sets. Hit-rate Recall@k, MRR, nDCG@10."""
    r = {f"R@{k}": 0.0 for k in ks}
    mrr = ndcg = 0.0
    for ranked, pos in zip(rank_lists, positives):
        first = next((j for j, i in enumerate(ranked) if i in pos), None)
        if first is not None:
            mrr += 1.0 / (first + 1)
            for k in ks:
                r[f"R@{k}"] += first < k
        dcg = sum(1.0 / np.log2(j + 2) for j, i in enumerate(ranked[:10]) if i in pos)
        idcg = sum(1.0 / np.log2(j + 2) for j in range(min(len(pos), 10)))
        ndcg += dcg / idcg if idcg else 0.0
    n = max(len(rank_lists), 1)
    out = {k: v / n for k, v in r.items()}
    out["MRR"], out["nDCG@10"] = mrr / n, ndcg / n
    return out


if __name__ == "__main__":  # smoke check
    m = metrics([[3, 1, 2], [9, 8, 7]], [{1}, {5}])
    assert m["R@1"] == 0 and m["R@5"] == 0.5 and abs(m["MRR"] - 0.25) < 1e-9, m
    s1, s2 = split_ids(range(100), 0), split_ids(range(100), 0)
    assert s1 == s2 and len(s1["train"]) == 80 and not set(s1["train"]) & set(s1["test"])
    print("common ok")

In [ ]:
%%writefile /kaggle/working/src/features.py
"""Frozen CLIP features (open_clip) + a fake encoder for CPU smoke tests.

Real:  Encoder("clip", img_dirs=[...])   Fake: Encoder("fake", images=images)
Both give .images(ids) -> (kept_ids, [n,d] L2-normed) and .texts(list) -> [m,d] L2-normed.
"""
import hashlib
import os

import numpy as np


def _unit(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-9)


class Encoder:
    def __init__(self, mode="clip", img_dirs=(), images=None, model="ViT-B-32", pretrained="laion2b_s34b_b79k",
                 device=None, batch=256, workers=4):
        self.mode, self.img_dirs, self.images_meta = mode, list(img_dirs), images
        if mode == "clip":
            import open_clip
            import torch
            self.torch = torch
            self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
            self.model, _, self.pre = open_clip.create_model_and_transforms(model, pretrained=pretrained)
            self.model = self.model.to(self.device).eval()
            if self.device == "cuda":
                self.model = self.model.half()
            self.tok = open_clip.get_tokenizer(model)
            self.batch, self.workers = batch, workers

    # ---- fake: deterministic word vectors, images = mean of their object words + noise
    def _w(self, word, d=64):
        h = int(hashlib.md5(word.encode()).hexdigest()[:8], 16)
        return np.random.RandomState(h).randn(d)

    def _fake_text(self, t):
        return np.mean([self._w(w) for w in t.split()], axis=0)

    def texts(self, items):
        if self.mode == "fake":
            return _unit(np.stack([self._fake_text(t) for t in items]).astype("float32"))
        out = []
        with self.torch.no_grad():
            for i in range(0, len(items), 512):
                tk = self.tok(items[i:i + 512]).to(self.device)
                out.append(self.model.encode_text(tk).float().cpu().numpy())
        return _unit(np.concatenate(out))

    def _find(self, iid):
        for d in self.img_dirs:
            p = os.path.join(d, f"{iid}.jpg")
            if os.path.exists(p):
                return p
        return None

    def images(self, ids):
        if self.mode == "fake":
            rs = np.random.RandomState(0)
            f = [np.mean([self._w(w) for w in self.images_meta[i]["objs"]] or [np.zeros(64)], axis=0)
                 + 0.5 * rs.randn(64) for i in ids]
            return list(ids), _unit(np.stack(f).astype("float32"))
        from PIL import Image
        torch = self.torch
        paths = [(i, self._find(i)) for i in ids]
        paths = [(i, p) for i, p in paths if p]
        print(f"images found: {len(paths)}/{len(ids)}")

        class DS(torch.utils.data.Dataset):
            def __len__(s):
                return len(paths)

            def __getitem__(s, k):
                return self.pre(Image.open(paths[k][1]).convert("RGB"))

        dl = torch.utils.data.DataLoader(DS(), batch_size=self.batch, num_workers=self.workers)
        out = []
        with torch.no_grad():
            for x in dl:
                x = x.to(self.device)
                x = x.half() if self.device == "cuda" else x
                out.append(self.model.encode_image(x).float().cpu().numpy())
        return [i for i, _ in paths], _unit(np.concatenate(out))

In [ ]:
%%writefile /kaggle/working/src/graphrank.py
"""CLIP + Graph reranking (Level 1, and the hook point for the Level 2 method).

Graph (leak-safe by construction):
  nodes  = concepts (object names) + images
  edges  = image--concept  "contains"  (object LIST of every image, no relations: input, not label)
           concept--concept relation edges, counted from TRAIN images only
So a val/test image never exposes its own relations; the model must infer "child plays football"
from concept relations learned on train. Ablations act on the concept--concept edges.

Score = a*z(cosine_clip) + (1-a)*z(graph_score), z = per-query z-score over the gallery, a tuned on val.
"""
import random
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from common import build_queries


def build_vocab(images, train_ids, min_count=3):
    c = Counter(o for i in train_ids for o in images[i]["objs"])
    return sorted(o for o, n in c.items() if n >= min_count)


def cc_edges(images, train_ids, vocab, min_edge=2):
    """Concept pairs linked by a relation in train; returns edge array [E,2] and {(a,b): Counter(pred)}."""
    idx = {c: k for k, c in enumerate(vocab)}
    cnt, preds = Counter(), defaultdict(Counter)
    for i in train_ids:
        seen = set()
        for s, p, o in images[i]["rels"]:
            if s in idx and o in idx and s != o:
                preds[(s, o)][p] += 1
                seen.add((idx[s], idx[o]))
        cnt.update(seen)
    e = np.array([k for k, n in cnt.items() if n >= min_edge], dtype=np.int64).reshape(-1, 2)
    return e, preds


def ablate(e, n_concepts, mode, seed):
    rng = np.random.RandomState(seed)
    if mode == "none" or len(e) == 0:
        return e
    if mode == "drop":
        return e[:0]
    if mode == "shuffle":  # keep sources, permute targets: same sizes, relational structure destroyed
        return np.stack([e[:, 0], rng.permutation(e[:, 1])], 1)
    if mode == "random":  # same number of edges, random endpoints
        return rng.randint(0, n_concepts, size=e.shape)
    raise ValueError(mode)


def build_adj(images, img_ids, vocab, cc, device):
    """Row-normalised (mean) sparse adjacency over [concepts | images], symmetric."""
    C = len(vocab)
    idx = {c: k for k, c in enumerate(vocab)}
    rows, cols = [], []
    for j, i in enumerate(img_ids):
        for o in images[i]["objs"]:
            if o in idx:
                rows.append(C + j)
                cols.append(idx[o])
    ic = np.array([rows, cols], dtype=np.int64).reshape(2, -1)
    ei = np.concatenate([ic, ic[::-1], cc.T, cc.T[::-1]], axis=1)
    N = C + len(img_ids)
    deg = np.bincount(ei[0], minlength=N).clip(min=1)
    val = 1.0 / deg[ei[0]]
    A = torch.sparse_coo_tensor(torch.tensor(ei), torch.tensor(val, dtype=torch.float32), (N, N))
    return A.coalesce().to(device)


class SAGE(nn.Module):
    """GraphSAGE-mean, 2 layers: h' = relu(W_s h + W_n mean_nbr(h)). Plain torch, no PyG needed."""

    def __init__(self, d_in, d_h=256, d_out=256):
        super().__init__()
        self.s1, self.n1 = nn.Linear(d_in, d_h), nn.Linear(d_in, d_h, bias=False)
        self.s2, self.n2 = nn.Linear(d_h, d_out), nn.Linear(d_h, d_out, bias=False)

    def forward(self, x, A):
        h = F.relu(self.s1(x) + self.n1(torch.sparse.mm(A, x)))
        return self.s2(h) + self.n2(torch.sparse.mm(A, h))


def train_gnn(X, A, img_off, img_pos, Q, queries, seed, d_q, epochs=60, bs=256, lr=1e-3, tau=0.07, device="cpu"):
    """InfoNCE: projected query text vs GNN image embedding. queries: list of dicts with 'pos' (global img indices)."""
    torch.manual_seed(seed)
    rng = random.Random(seed)
    gnn, proj = SAGE(X.shape[1]).to(device), nn.Linear(d_q, 256).to(device)
    opt = torch.optim.Adam(list(gnn.parameters()) + list(proj.parameters()), lr=lr)
    n = len(queries)
    for ep in range(epochs):
        order = list(range(n))
        rng.shuffle(order)
        tot = 0.0
        for b in range(0, n, bs):
            bi = order[b:b + bs]
            H = F.normalize(gnn(X, A)[img_off:], dim=-1)
            q = F.normalize(proj(Q[bi]), dim=-1)
            pos = torch.tensor([rng.choice(queries[k]["pos_idx"]) for k in bi], device=device)
            logits = q @ H[pos].T / tau
            mask = torch.zeros_like(logits, dtype=torch.bool)  # other queries' positives are not negatives
            for r, k in enumerate(bi):
                ps = set(queries[k]["pos_idx"])
                for c, p in enumerate(pos.tolist()):
                    if p in ps and c != r:
                        mask[r, c] = True
            logits = logits.masked_fill(mask, -1e9)
            loss = F.cross_entropy(logits, torch.arange(len(bi), device=device))
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item()
    return gnn, proj


def z(s):
    return (s - s.mean(1, keepdims=True)) / (s.std(1, keepdims=True) + 1e-9)


def rank_fused(clip_s, graph_s, alpha, topk=100):
    """Rerank top-k by CLIP using fused z-scores; the rest stays in CLIP order below them."""
    f = alpha * z(clip_s) + (1 - alpha) * z(graph_s)
    top = np.argsort(-clip_s, 1)[:, :topk]
    s = clip_s.astype("float64") - 1000.0  # outside top-k: CLIP order, below every reranked item
    for r in range(len(s)):
        s[r, top[r]] = f[r, top[r]]
    return np.argsort(-s, 1)


def explain(query_triple, image_objs, preds, top=3):
    """Concept relations connecting the query's subject/object to the image's objects (1 and 2 hops)."""
    s, p, o = query_triple
    nbr = defaultdict(list)
    for (a, b), c in preds.items():
        nbr[a].append((b, c))
        nbr[b].append((a, c))
    out = []
    for q in (s, o):
        for x, c in nbr.get(q, []):
            if x in image_objs and x != q:
                out.append((sum(c.values()), f"{q} --[{c.most_common(1)[0][0]}]-- {x}"))
    return [t for _, t in sorted(out, reverse=True)[:top]]

In [ ]:
%%writefile /kaggle/working/src/export_vg_mmkg.py
"""Export a Visual Genome subset as an MMKG in AdaMF-MAT (OpenKE) format  (Level 2, plan A).

Entities  : concepts (object names, vocab from TRAIN images) + images.        names: c:<name>, img:<id>
Relations : "contains" (image -> concept) + the top-P predicates (concept -> concept).
train2id  : (image, contains, concept) for ALL images (object list = input, not label)
            + concept relation triples seen in >= min_triple TRAIN images only.
valid/test: concept relation triples that appear only in val/test images and not in train (new facts).
Embeddings (CLIP, 512-d): visual  = image feature (concept = mean of TRAIN images containing it)
                          textual = text feature (concept name; image = mean of its concepts' text features)
Leak rules: eval-query positive images (data/eval_queries.json) are forced into TEST, so their relations never reach training.

Usage (Kaggle, after cloning the repo to /kaggle/working/AdaMF-MAT):
  python src/export_vg_mmkg.py --scene .../scene_graphs.json --img-dirs D1 D2 --n 20000 --repo /kaggle/working/AdaMF-MAT \
      --eval-queries data/eval_queries.json
Local structure test:  python src/export_vg_mmkg.py --scene data/subset_5k.json --fake --repo /tmp/x --skip-nn
"""
import argparse
import json
import os
import random
import shutil
import subprocess
from collections import Counter, defaultdict

import numpy as np

from common import load_images, split_ids
from features import Encoder


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--scene", required=True)
    ap.add_argument("--img-dirs", nargs="*", default=[])
    ap.add_argument("--n", type=int, default=None)
    ap.add_argument("--split-seed", type=int, default=0)
    ap.add_argument("--repo", required=True, help="AdaMF-MAT repo dir; files go to <repo>/benchmarks/<name>, <repo>/embeddings")
    ap.add_argument("--name", default="VG")
    ap.add_argument("--min-count", type=int, default=3)
    ap.add_argument("--min-triple", type=int, default=2)
    ap.add_argument("--top-pred", type=int, default=30)
    ap.add_argument("--eval-queries", default=None)
    ap.add_argument("--fake", action="store_true")
    ap.add_argument("--skip-nn", action="store_true", help="skip running the repo's n-n.py (needed on Kaggle)")
    a = ap.parse_args()

    eval_ids = set()
    if a.eval_queries:
        eval_ids = {i for q in json.load(open(a.eval_queries)) for i in q["positives"]}
    images = load_images(a.scene, a.n, a.split_seed)
    if eval_ids:  # make sure the eval images are present even if the random subset missed them
        images.update(load_images(a.scene, None, a.split_seed, keep_ids=eval_ids - set(images)))
    enc = Encoder("fake", images=images) if a.fake else Encoder("clip", img_dirs=a.img_dirs)
    kept, IF = enc.images(sorted(images))
    images = {i: images[i] for i in kept}
    pos_of = {i: k for k, i in enumerate(kept)}

    sp = split_ids(kept, a.split_seed)
    ev = eval_ids & set(kept)  # force eval positives into test
    sp["train"] = [i for i in sp["train"] if i not in ev]
    sp["val"] = [i for i in sp["val"] if i not in ev]
    sp["test"] = sorted(set(sp["test"]) | ev)
    print({k: len(v) for k, v in sp.items()}, "images;", len(ev), "eval images forced to test")

    # vocab + predicates from TRAIN only
    cnt = Counter(o for i in sp["train"] for o in images[i]["objs"])
    vocab = sorted(o for o, c in cnt.items() if c >= a.min_count)
    cidx = {c: k for k, c in enumerate(vocab)}
    pc = Counter(p for i in sp["train"] for _, p, _ in images[i]["rels"])
    preds = [p for p, _ in pc.most_common(a.top_pred)]
    print(f"{len(vocab)} concepts, {len(preds)} predicates")

    def triples(ids):
        c = Counter()
        for i in ids:
            for s, p, o in images[i]["rels"]:
                if s in cidx and o in cidx and s != o and p in preds:
                    c[(s, p, o)] += 1
        return c

    tr = triples(sp["train"])
    train_rel = {t for t, c in tr.items() if c >= a.min_triple}
    heldout = sorted(t for t in triples(sp["val"] + sp["test"]) if t not in tr)
    random.Random(a.split_seed).shuffle(heldout)
    half = len(heldout) // 2
    valid_rel, test_rel = heldout[:half], heldout[half:]
    if not test_rel:  # tiny subsets: the repo's tester needs >= 1 test triple
        test_rel = sorted(train_rel)[:1]
    print(f"concept triples: train {len(train_rel)}, valid {len(valid_rel)}, test {len(test_rel)}")

    # entities / relations
    C = len(vocab)
    ents = [f"c:{c.replace(' ', '_')}" for c in vocab] + [f"img:{i}" for i in kept]
    rels = ["contains"] + [p.replace(" ", "_") for p in preds]
    ridx = {p: k + 1 for k, p in enumerate(preds)}
    bdir = os.path.join(a.repo, "benchmarks", a.name)
    os.makedirs(bdir, exist_ok=True)
    os.makedirs(os.path.join(a.repo, "embeddings"), exist_ok=True)

    def write(fname, rows):  # OpenKE: first line = count, then "h t r"
        with open(os.path.join(bdir, fname), "w") as f:
            f.write(f"{len(rows)}\n")
            for h, t, r in rows:
                f.write(f"{h} {t} {r}\n")

    contains = [(C + pos_of[i], cidx[o], 0) for i in kept for o in sorted(images[i]["objs"]) if o in cidx]
    tr_rows = contains + [(cidx[s], cidx[o], ridx[p]) for s, p, o in sorted(train_rel)]
    write("train2id.txt", tr_rows)
    write("valid2id.txt", [(cidx[s], cidx[o], ridx[p]) for s, p, o in valid_rel])
    write("test2id.txt", [(cidx[s], cidx[o], ridx[p]) for s, p, o in test_rel])
    for fname, names in (("entity2id.txt", ents), ("relation2id.txt", rels)):
        with open(os.path.join(bdir, fname), "w") as f:
            f.write(f"{len(names)}\n")
            for k, n in enumerate(names):
                f.write(f"{n} {k}\n")
    print(f"{len(ents)} entities, {len(rels)} relations, {len(tr_rows)} train triples ({len(contains)} contains)")

    # embeddings
    import torch
    T = enc.texts([f"a photo of a {c}" for c in vocab])  # [C,d]
    d = T.shape[1]
    vis = np.zeros((len(ents), d), "float32")
    txt = np.zeros((len(ents), d), "float32")
    txt[:C] = T
    sums, ns = np.zeros((C, d), "float32"), np.zeros(C)
    for i in sp["train"]:
        for o in images[i]["objs"]:
            if o in cidx:
                sums[cidx[o]] += IF[pos_of[i]]
                ns[cidx[o]] += 1
    vis[:C] = sums / np.maximum(ns, 1)[:, None]
    for i in kept:
        k = C + pos_of[i]
        vis[k] = IF[pos_of[i]]
        cs = [cidx[o] for o in images[i]["objs"] if o in cidx]
        if cs:
            m = T[cs].mean(0)
            txt[k] = m / (np.linalg.norm(m) + 1e-9)
    emb = os.path.join(a.repo, "embeddings")
    torch.save(torch.tensor(vis), f"{emb}/{a.name}-visual.pth")
    torch.save(torch.tensor(txt), f"{emb}/{a.name}-textual.pth")

    meta = {"name": a.name, "vocab": vocab, "preds": preds, "n_concepts": C, "kept": kept, "split": sp,
            "eval_images": sorted(ev), "dim": int(d)}
    json.dump(meta, open(os.path.join(a.repo, f"{a.name}_meta.json"), "w"))

    if not a.skip_nn:  # repo's own script writes type_constrain.txt, 1-1.txt ... test2id_all.txt
        src = os.path.join(a.repo, "benchmarks", "MKG-W", "n-n.py")
        shutil.copy(src, os.path.join(bdir, "n-n.py"))
        subprocess.run(["python", "n-n.py"], cwd=bdir, check=True)
    print("export done ->", a.repo)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/src/run_level2.py
"""Level 2 evaluation: CLIP vs CLIP + AdaMF-MAT scorer on the 24 eval queries (direct vs indirect).

Run on Kaggle AFTER export_vg_mmkg.py and training AdaMF-MAT on the exported dataset:
  python src/run_level2.py --repo /kaggle/working/AdaMF-MAT --ckpt checkpoint/VG-full --scene <scene_graphs.json> \
      --eval-queries data/eval_queries.json

Graph score of image i for query (s, p, o) = sum over query concepts c in {s, o} (that are in the vocab) of
AdaMF-MAT plausibility of the triple (image_i, contains, c). Relation knowledge reaches images through the
concept--concept triples learned from TRAIN images. Fusion = a*z(clip) + (1-a)*z(graph), a picked on VAL queries.
"""
import argparse
import json
import os
import sys

import numpy as np
import torch

from common import build_queries, load_images, metrics
from features import Encoder
from graphrank import rank_fused

ap = argparse.ArgumentParser()
ap.add_argument("--repo", required=True)
ap.add_argument("--ckpt", required=True, help="checkpoint path relative to repo, e.g. checkpoint/VG-full")
ap.add_argument("--scene", required=True)
ap.add_argument("--eval-queries", required=True)
ap.add_argument("--name", default="VG")
ap.add_argument("--dim", type=int, default=200)
ap.add_argument("--margin", type=float, default=12.0)
ap.add_argument("--topk", type=int, default=100)
ap.add_argument("--out", default="results_level2.json")
a = ap.parse_args()

sys.path.insert(0, a.repo)
from mmkgc.module.model import AdvMixRotatE  # repo code, license not checked: do not redistribute

dev = "cuda" if torch.cuda.is_available() else "cpu"
meta = json.load(open(os.path.join(a.repo, f"{a.name}_meta.json")))
vocab, C, kept, sp = meta["vocab"], meta["n_concepts"], meta["kept"], meta["split"]
cidx = {c: k for k, c in enumerate(vocab)}
pos_of = {i: k for k, i in enumerate(kept)}
img_emb = torch.load(os.path.join(a.repo, "embeddings", f"{a.name}-visual.pth"))
txt_emb = torch.load(os.path.join(a.repo, "embeddings", f"{a.name}-textual.pth"))
IF = img_emb[C:].numpy()  # CLIP image features, row j = kept[j]

ent_tot = int(open(os.path.join(a.repo, "benchmarks", a.name, "entity2id.txt")).readline())
rel_tot = int(open(os.path.join(a.repo, "benchmarks", a.name, "relation2id.txt")).readline())
model = AdvMixRotatE(ent_tot=ent_tot, rel_tot=rel_tot, dim=a.dim, margin=a.margin, epsilon=2.0,
                     img_emb=img_emb, text_emb=txt_emb)
model.load_state_dict(torch.load(os.path.join(a.repo, a.ckpt), map_location="cpu"))
model = model.to(dev).eval()
enc = Encoder("clip")  # text side only


def graph_scores(queries, ids):
    """[n_queries, n_ids] AdaMF-MAT score of (image, contains, concept) summed over the query's concepts."""
    ent = torch.tensor([C + pos_of[i] for i in ids], device=dev)
    out = np.zeros((len(queries), len(ids)), "float32")
    cache = {}
    with torch.no_grad():
        for qi, q in enumerate(queries):
            for c in {q["triple"][0], q["triple"][2]}:
                if c not in cidx:
                    continue
                if c not in cache:
                    h = ent
                    t = torch.full_like(h, cidx[c])
                    r = torch.zeros_like(h)
                    cache[c] = model({"batch_h": h, "batch_t": t, "batch_r": r, "mode": "normal"}).float().cpu().numpy()
                out[qi] += cache[c]
    return out


def evaluate(queries, ids, alpha):
    clip_s = enc.texts([q["text"] for q in queries]) @ IF[[pos_of[i] for i in ids]].T
    g = graph_scores(queries, ids)
    ranks = rank_fused(clip_s, g, alpha, a.topk)
    ids = np.array(ids)
    pos = [set(q["positives"]) & set(ids.tolist()) for q in queries]
    return metrics([list(ids[r]) for r in ranks], pos)


# val queries (annotation triples of val images) pick alpha; test gallery + eval queries report
val_img = load_images(a.scene, None, 0, keep_ids=set(sp["val"]))
val_ids = [i for i in sp["val"] if i in val_img]
val_q = build_queries(val_img, val_ids)
alphas = [round(x, 1) for x in np.arange(0, 1.01, 0.1)]
val_mrr = {al: evaluate(val_q, val_ids, al)["MRR"] for al in alphas}
best = max(alphas, key=lambda al: val_mrr[al])
print(f"{len(val_q)} val queries; alpha per val MRR: {val_mrr}; chosen alpha = {best}")

eq = json.load(open(a.eval_queries))
test_ids = sp["test"]
res = {"alpha": best, "n_val_queries": len(val_q)}
for name, qs in (("direct", [q for q in eq if q["group"] == "direct"]),
                 ("indirect", [q for q in eq if q["group"] == "indirect"]),
                 ("all", eq)):
    res[name] = {"clip": evaluate(qs, test_ids, 1.0),
                 "adamf_only": evaluate(qs, test_ids, 0.0),
                 "clip+adamf": evaluate(qs, test_ids, best)}
    print(f"== {name} ({len(qs)} queries)")
    for k, v in res[name].items():
        print(f"  {k:11}", {m: round(float(x), 4) for m, x in v.items()})
json.dump(res, open(a.out, "w"), indent=1, default=float)
print("saved", a.out)

In [ ]:
%%writefile /kaggle/working/src/eval_queries.json
[
 {
  "id": "i0",
  "group": "indirect",
  "text": "woman holding boy",
  "triple": [
   "woman",
   "holding",
   "boy"
  ],
  "positives": [
   2315429,
   2316603,
   2321127,
   2321419,
   2321884,
   2329310,
   2332907,
   2336989,
   2342383,
   2344827,
   2362032,
   2363931,
   2364731,
   2369876,
   2375896,
   2380332,
   2380800,
   2383374,
   2389343,
   2391582,
   2393352,
   2395598,
   2395848,
   2398854,
   2402103,
   2411527,
   2414483,
   2414926
  ],
  "examples": [
   {
    "image": 2375896,
    "via": "umbrella"
   },
   {
    "image": 2393352,
    "via": "hat"
   },
   {
    "image": 2402103,
    "via": "face"
   },
   {
    "image": 2344827,
    "via": "shadow"
   },
   {
    "image": 2336989,
    "via": "hair"
   }
  ],
  "check": ""
 },
 {
  "id": "i1",
  "group": "indirect",
  "text": "person sitting on horse",
  "triple": [
   "person",
   "sitting on",
   "horse"
  ],
  "positives": [
   2320662,
   2332304,
   2335912,
   2342912,
   2343707,
   2355548,
   2363819,
   2367271,
   2367691,
   2372673,
   2377364,
   2378560,
   2382213,
   2383628,
   2387746,
   2403436,
   2403572,
   2410874,
   2411183
  ],
  "examples": [
   {
    "image": 2342912,
    "via": "fence"
   },
   {
    "image": 2367691,
    "via": "saddle"
   },
   {
    "image": 2320662,
    "via": "carriage"
   },
   {
    "image": 2355548,
    "via": "carriage"
   },
   {
    "image": 2410874,
    "via": "wagon"
   }
  ],
  "check": ""
 },
 {
  "id": "i2",
  "group": "indirect",
  "text": "truck pulling trailer",
  "triple": [
   "truck",
   "pulling",
   "trailer"
  ],
  "positives": [
   150385,
   1159732,
   2332544,
   2345316,
   2371775,
   2377245,
   2396252,
   2397683,
   2409644,
   2412052
  ],
  "examples": [
   {
    "image": 2412052,
    "via": "canoes"
   },
   {
    "image": 2377245,
    "via": "boat"
   },
   {
    "image": 1159732,
    "via": "boat"
   },
   {
    "image": 2396252,
    "via": "target truck"
   },
   {
    "image": 2409644,
    "via": "plate"
   }
  ],
  "check": ""
 },
 {
  "id": "i3",
  "group": "indirect",
  "text": "person holding leash",
  "triple": [
   "person",
   "holding",
   "leash"
  ],
  "positives": [
   308,
   3360,
   713697,
   2318115,
   2318965,
   2321507,
   2330686,
   2331644,
   2337587,
   2343433,
   2365524,
   2367250,
   2367662,
   2373693,
   2376077,
   2380294,
   2402079
  ],
  "examples": [
   {
    "image": 2380294,
    "via": "suitcase"
   },
   {
    "image": 2343433,
    "via": "dog"
   },
   {
    "image": 2373693,
    "via": "dog"
   },
   {
    "image": 2330686,
    "via": "dog"
   },
   {
    "image": 2365524,
    "via": "dog"
   }
  ],
  "check": ""
 },
 {
  "id": "i4",
  "group": "indirect",
  "text": "man pulling luggage",
  "triple": [
   "man",
   "pulling",
   "luggage"
  ],
  "positives": [
   713383,
   1593095,
   2322469,
   2334171,
   2364116,
   2373301,
   2374380,
   2407749,
   2411806,
   2412870
  ],
  "examples": [
   {
    "image": 2322469,
    "via": "woman"
   },
   {
    "image": 2412870,
    "via": "cart"
   },
   {
    "image": 713383,
    "via": "cart"
   },
   {
    "image": 2364116,
    "via": "backpack"
   },
   {
    "image": 2374380,
    "via": "person"
   }
  ],
  "check": ""
 },
 {
  "id": "i5",
  "group": "indirect",
  "text": "person playing with snowboard",
  "triple": [
   "person",
   "playing with",
   "snowboard"
  ],
  "positives": [
   2335527,
   2335553,
   2346391,
   2355081,
   2366614,
   2367083,
   2370670,
   2381191,
   2384371,
   2413381
  ],
  "examples": [
   {
    "image": 2384371,
    "via": "blue"
   },
   {
    "image": 2335527,
    "via": "foot"
   },
   {
    "image": 2335553,
    "via": "boy"
   },
   {
    "image": 2413381,
    "via": "foot"
   },
   {
    "image": 2370670,
    "via": "snow"
   }
  ],
  "check": ""
 },
 {
  "id": "i6",
  "group": "indirect",
  "text": "person carrying luggage",
  "triple": [
   "person",
   "carrying",
   "luggage"
  ],
  "positives": [
   2373301,
   2381177,
   2385218,
   2394257,
   2410094
  ],
  "examples": [
   {
    "image": 2410094,
    "via": "seat"
   },
   {
    "image": 2394257,
    "via": "luggage cart"
   },
   {
    "image": 2373301,
    "via": "horse"
   },
   {
    "image": 2385218,
    "via": "cart"
   },
   {
    "image": 2381177,
    "via": "truck"
   }
  ],
  "check": ""
 },
 {
  "id": "i7",
  "group": "indirect",
  "text": "person holding donut",
  "triple": [
   "person",
   "holding",
   "donut"
  ],
  "positives": [
   2349184,
   2354795,
   2380954,
   2393027,
   2410868
  ],
  "examples": [
   {
    "image": 2354795,
    "via": "bag"
   },
   {
    "image": 2349184,
    "via": "girl"
   },
   {
    "image": 2380954,
    "via": "hand"
   },
   {
    "image": 2410868,
    "via": "girl"
   },
   {
    "image": 2393027,
    "via": "hand"
   }
  ],
  "check": ""
 },
 {
  "id": "i8",
  "group": "indirect",
  "text": "horse carrying person",
  "triple": [
   "horse",
   "carrying",
   "person"
  ],
  "positives": [
   2322503,
   2329033,
   2342920,
   2357923,
   2366894,
   2391000,
   2417239
  ],
  "examples": [
   {
    "image": 2342920,
    "via": "head"
   },
   {
    "image": 2322503,
    "via": "head"
   },
   {
    "image": 2329033,
    "via": "carriage"
   },
   {
    "image": 2417239,
    "via": "head"
   },
   {
    "image": 2357923,
    "via": "man"
   }
  ],
  "check": ""
 },
 {
  "id": "i9",
  "group": "indirect",
  "text": "man standing on floor",
  "triple": [
   "man",
   "standing on",
   "floor"
  ],
  "positives": [
   10,
   420,
   4506,
   61525,
   1592775,
   2315755,
   2321830,
   2325698,
   2326626,
   2327619,
   2330166,
   2331770,
   2333797,
   2335177,
   2335650,
   2337945,
   2345583,
   2346208,
   2347158,
   2349452,
   2349948,
   2350522,
   2351216,
   2354234,
   2354555,
   2354594,
   2354691,
   2354948,
   2355374,
   2355841,
   2355880,
   2361758,
   2362454,
   2366083,
   2369170,
   2370203,
   2372743,
   2382465,
   2382644,
   2386511,
   2387217,
   2389719,
   2393006,
   2400031,
   2403927,
   2404478,
   2404576,
   2404760,
   2405161,
   2405245,
   2407580,
   2407749,
   2408253,
   2408875,
   2409779,
   2410049
  ],
  "examples": [
   {
    "image": 10,
    "via": "coat"
   },
   {
    "image": 2326626,
    "via": "luggage"
   },
   {
    "image": 2351216,
    "via": "suitcase"
   },
   {
    "image": 420,
    "via": "cup"
   },
   {
    "image": 2408875,
    "via": "kitchen"
   }
  ],
  "check": ""
 },
 {
  "id": "i10",
  "group": "indirect",
  "text": "cat sitting on bed",
  "triple": [
   "cat",
   "sitting on",
   "bed"
  ],
  "positives": [
   498281,
   2317863,
   2317872,
   2325026,
   2328620,
   2345257,
   2350721,
   2354145,
   2357079,
   2358783,
   2360615,
   2364706,
   2364843,
   2365482,
   2365734,
   2371517,
   2371658,
   2374117,
   2377610,
   2377840,
   2378018,
   2381472,
   2403463,
   2412395,
   2414640,
   2417720
  ],
  "examples": [
   {
    "image": 2365482,
    "via": "person"
   },
   {
    "image": 2328620,
    "via": "laptop"
   },
   {
    "image": 2414640,
    "via": "suitcase"
   },
   {
    "image": 2371658,
    "via": "baby"
   },
   {
    "image": 2377840,
    "via": "suitcase"
   }
  ],
  "check": ""
 },
 {
  "id": "i11",
  "group": "indirect",
  "text": "broccoli sitting on plate",
  "triple": [
   "broccoli",
   "sitting on",
   "plate"
  ],
  "positives": [
   285815,
   2317613,
   2318236,
   2318725,
   2319789,
   2330814,
   2331432,
   2333416,
   2333990,
   2335090,
   2336112,
   2337339,
   2338160,
   2339426,
   2342321,
   2343284,
   2349599,
   2350650,
   2352389,
   2356135,
   2363330,
   2365120,
   2369949,
   2371700,
   2372380,
   2373095,
   2374947,
   2383265,
   2383278,
   2385326,
   2392794,
   2400278,
   2402125,
   2402692,
   2402718,
   2406815,
   2410663,
   2417304
  ],
  "examples": [
   {
    "image": 2400278,
    "via": "table"
   },
   {
    "image": 2371700,
    "via": "pizza"
   },
   {
    "image": 2410663,
    "via": "gravy"
   },
   {
    "image": 2330814,
    "via": "meal"
   },
   {
    "image": 2335090,
    "via": "pizza"
   }
  ],
  "check": ""
 },
 {
  "id": "d0",
  "group": "direct",
  "text": "zebra standing on field",
  "triple": [
   "zebra",
   "standing on",
   "field"
  ],
  "positives": [
   2331531,
   2342648,
   2345091,
   2359738,
   2372068,
   2395436
  ],
  "examples": [],
  "check": ""
 },
 {
  "id": "d1",
  "group": "direct",
  "text": "boy eating food",
  "triple": [
   "boy",
   "eating",
   "food"
  ],
  "positives": [
   2318034,
   2326204,
   2335709,
   2358723,
   2385528,
   2388790,
   2400651,
   2410103,
   2413148,
   2414822
  ],
  "examples": [],
  "check": ""
 },
 {
  "id": "d2",
  "group": "direct",
  "text": "hand feeding giraffe",
  "triple": [
   "hand",
   "feeding",
   "giraffe"
  ],
  "positives": [
   2318576,
   2333571,
   2345006,
   2362739,
   2377332,
   2384285,
   2384903,
   2389627,
   2407383,
   2409072
  ],
  "examples": [],
  "check": ""
 },
 {
  "id": "d3",
  "group": "direct",
  "text": "people standing on street",
  "triple": [
   "people",
   "standing on",
   "street"
  ],
  "positives": [
   150386,
   498373,
   713828,
   2335471,
   2344192,
   2348024,
   2348696,
   2348954,
   2371338,
   2377328,
   2382057,
   2392026,
   2398467,
   2401794,
   2409336
  ],
  "examples": [],
  "check": ""
 },
 {
  "id": "d4",
  "group": "direct",
  "text": "woman walking on sand",
  "triple": [
   "woman",
   "walking on",
   "sand"
  ],
  "positives": [
   2315786,
   2335060,
   2335728,
   2341351,
   2395612,
   2397927,
   2401128,
   2401604
  ],
  "examples": [],
  "check": ""
 },
 {
  "id": "d5",
  "group": "direct",
  "text": "cow eating hay",
  "triple": [
   "cow",
   "eating",
   "hay"
  ],
  "positives": [
   2343727,
   2346054,
   2346590,
   2358701,
   2378820,
   2380297,
   2385642,
   2385723,
   2385775,
   2389824,
   2399999,
   2400837,
   2415434
  ],
  "examples": [],
  "check": ""
 },
 {
  "id": "d6",
  "group": "direct",
  "text": "person driving bus",
  "triple": [
   "person",
   "driving",
   "bus"
  ],
  "positives": [
   713021,
   2315504,
   2325048,
   2338031,
   2338634,
   2345911,
   2346044,
   2356627,
   2356640,
   2360873,
   2362224,
   2371964,
   2394994,
   2396921
  ],
  "examples": [],
  "check": ""
 },
 {
  "id": "d7",
  "group": "direct",
  "text": "elephant carrying man",
  "triple": [
   "elephant",
   "carrying",
   "man"
  ],
  "positives": [
   2325746,
   2343985,
   2352550,
   2357053,
   2358518,
   2360342,
   2377837,
   2396857,
   2414380
  ],
  "examples": [],
  "check": ""
 },
 {
  "id": "d8",
  "group": "direct",
  "text": "people watching person",
  "triple": [
   "people",
   "watching",
   "person"
  ],
  "positives": [
   2343415,
   2347842,
   2359949,
   2369087,
   2406016,
   2406039,
   2408089
  ],
  "examples": [],
  "check": ""
 },
 {
  "id": "d9",
  "group": "direct",
  "text": "people riding bus",
  "triple": [
   "people",
   "riding",
   "bus"
  ],
  "positives": [
   631,
   2315804,
   2341028,
   2367618,
   2389709,
   2404657,
   2406978,
   2411869
  ],
  "examples": [],
  "check": ""
 },
 {
  "id": "d10",
  "group": "direct",
  "text": "people walking on walkway",
  "triple": [
   "people",
   "walking on",
   "walkway"
  ],
  "positives": [
   2318007,
   2335287,
   2338990,
   2369276,
   2373288,
   2402531
  ],
  "examples": [],
  "check": ""
 },
 {
  "id": "d11",
  "group": "direct",
  "text": "boy watching tv",
  "triple": [
   "boy",
   "watching",
   "tv"
  ],
  "positives": [
   2317480,
   2347492,
   2353147,
   2390645,
   2407668
  ],
  "examples": [],
  "check": ""
 }
]

In [ ]:
@stage("2 export Visual Genome as an MMKG (CLIP features, real images)")
def _():
    cmd = (f"python export_vg_mmkg.py --scene {SCENE} --img-dirs {' '.join(IMGS)} --n {N_IMAGES} "
           f"--repo {R} --eval-queries {SRC}/eval_queries.json")
    rc, o = sh(cmd, cwd=SRC, timeout=7200)
    log("rc", rc, o)
    if rc != 0: STATE["ok"] = False

In [ ]:
def train(epochs, tag, timeout_s=None):
    os.makedirs(f"{R}/checkpoint", exist_ok=True)
    logf = f"{OUT}/VG-{tag}.log"
    tmo = f"timeout {timeout_s} " if timeout_s else ""
    cmd = (f"CUDA_VISIBLE_DEVICES=0 {tmo}python run_adamf_mat.py -dataset=VG -batch_size=1024 -margin=12 "
           f"-epoch={epochs} -dim=200 -mu=0 -save=./checkpoint/VG-{tag} -neg_num=128 -learning_rate={LR} > {logf} 2>&1")
    t = time.time(); rc = subprocess.run(cmd, shell=True, cwd=R).returncode
    log(f"VG {tag}: rc={rc} time={time.time()-t:.0f}s\n--- log tail ---\n{open(logf).read()[-2500:]}")
    return rc, logf

@stage("3 speed test: 2 epochs on the exported VG data (also checks that the repo accepts the files)")
def _():
    if not STATE["ok"]: log("skipped (earlier failure)"); return
    rc, logf = train(2, "smoke", 3600)
    if rc != 0: STATE["ok"] = False; return
    sp = re.findall(r"([\d.]+)s/it", open(logf).read())
    sec = float(sp[-1]); STATE["epochs"] = max(2, min(EPOCH_CAP, int(BUDGET_H * 3600 / sec)))
    log(f"{sec:.1f} s/epoch -> training {STATE['epochs']} epochs (budget {BUDGET_H} h)")

In [ ]:
@stage("4 train AdaMF-MAT on Visual Genome")
def _():
    if not STATE["ok"] or not STATE["epochs"]: log("skipped (earlier failure)"); return
    rc, _ = train(STATE["epochs"], "full", int((BUDGET_H + 1) * 3600))
    if rc != 0: STATE["ok"] = False

In [ ]:
@stage("5 evaluate: CLIP vs CLIP+AdaMF-MAT on the 24 queries (direct / indirect)")
def _():
    if not STATE["ok"]: log("skipped (earlier failure)"); return
    cmd = (f"python run_level2.py --repo {R} --ckpt checkpoint/VG-full --scene {SCENE} "
           f"--eval-queries {SRC}/eval_queries.json --out {OUT}/results_level2.json")
    rc, o = sh(cmd, cwd=SRC, timeout=3600)
    log("rc", rc, o)

In [ ]:
log("\nDONE. Send me /kaggle/working/report_level2.txt and results_level2.json (small files).")